# Experiment 44 — ML-1M Local-Credit Tournament

One short-horizon screening run over six diverse local-learning families, all from the **same SparseWalker initialization** and the same ML-1M protocol.

Arms:
1. `analytic_lc` — Experiment 43 strict backward-free baseline
2. `local_ce` — exact **local** sampled-CE gradient, recurrent state detached
3. `ff_goodness` — Forward-Forward-inspired positive/negative goodness
4. `target_denoise` — NoProp-inspired noisy target broadcast / denoising
5. `forward_gradient` — forward-only activation directional derivative of ranking loss
6. `vector_eligibility` — e-prop-inspired vector signal × eligibility trace

No test evaluation is used during the tournament. We rank arms by full-catalog validation NDCG@10, then take only promising families into deeper experiments.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('TORCH',torch.__version__,flush=True)
print('HEAD',HEAD,flush=True)
assert torch.cuda.is_available()


## Run tournament

Default is **3 epochs per arm**. That is intentional: Experiment 43 reached its best validation score by epoch 3, so this is a family-screening experiment rather than a convergence benchmark. Local-autograd arms are allowed to use reverse AD *inside the current detached block only*; no gradient crosses time or graph hops. Forward-only arms use no autograd.


In [ ]:
import runpy, os, sys
EPOCHS_PER_ARM=3
SCRIPT=f'{REPO}/experiments/run_ml1m_local_credit_tournament.py'
assert Path(SCRIPT).exists()
argv=[SCRIPT,
      '--epochs-per-arm',str(EPOCHS_PER_ARM),
      '--batch-size','512',
      '--eval-batch-size','1024',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data']
print('TOURNAMENT_IN_PROCESS',' '.join(argv),flush=True)
old_argv=sys.argv[:]; old_cwd=os.getcwd(); sys.argv=argv; os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv; os.chdir(old_cwd)


## Ranked result


In [ ]:
import json, pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_credit_tournament/seed42')
sp=root/'summary.json'
if sp.exists():
    summary=json.loads(sp.read_text())
    df=pd.DataFrame(summary['ranking'])
    display(df)
    print(json.dumps(summary,indent=2))
else:
    print('summary.json not written yet')
